# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, their fields, and display their `@id`s.

All entities are referenced by their Croissant `@id`.

In [ ]:
# Print all available record sets with their @id and names
print("Available Record Sets:")
for rset in dataset.record_sets:
    print(f"- {rset['@id']} (name: {rset.get('name', 'N/A')})")

# For this dataset, let's list fields for each record set
for rset in dataset.record_sets:
    print(f"\nFields for Record Set {rset['@id']}:")
    for field in rset['field']:
        print(f"  - {field['@id']} : {field.get('name', 'N/A')}")

## 3. Data Extraction
For demonstration, we load all available record sets into pandas DataFrames.

_Use record set and field `@id`s from the overview above as variables for all downstream code._

In [ ]:
# Get record set @id values
record_sets_ids = [rset['@id'] for rset in dataset.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    # List of dicts
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Show columns of the main (first) record set and preview
main_record_set_id = record_sets_ids[0]
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Below we apply a few standard data wrangling tasks on the main record set (using its `@id`).

- Select a numeric field for filtering and normalization by its `@id` (used as the DataFrame column name).
- Filter outliers, normalize a numeric field, and group by a categorical field.

In [ ]:
df = dataframes[main_record_set_id]

# List the numeric fields available
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric fields detected:", numeric_columns)

# Select one numeric field (by its true Croissant field @id / which will be the key/column)
if len(numeric_columns):
    numeric_field = numeric_columns[0]  # Pick the first numeric field
else:
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()  # Try using mean for threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records in '{main_record_set_id}' with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find the first object type column as a group field candidate
    group_fields = [col for col in df.columns if df[col].dtype == 'object']
    group_field = group_fields[0] if group_fields else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print('No numeric fields detected for EDA.')

## 5. Visualization
Let's plot the distribution of the selected numeric field and its normalized values.

In [ ]:
if numeric_field:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1, 2, 2)
    if f"{numeric_field}_normalized" in filtered_df:
        filtered_df[f"{numeric_field}_normalized"].hist(bins=30)
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.title(f"Normalized {numeric_field} Distribution (Filtered)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to reference and explore data using Croissant `@id`s. You can extend it with more advanced analysis, modeling, or linkage to external knowledge sources!
